# LAB 1 – Bài 5a, Trang 45: Market Basket – Germany

**Sinh viên:** Lê Nguyễn Anh Dự – **MSSV:** 6451071011

Lặp lại quy trình phân tích giỏ hàng trên Online Retail nhưng chỉ lấy khách hàng ở Germany. Dùng `min_sup=5%`, `min_conf=50%`.

**Thiết lập:** Chạy cell cài đặt/vá thư viện một lần, Restart session, rồi chạy các cell phân tích. Cảnh báo từ máy chủ Colab bên ngoài runtime không thể sửa bằng notebook.


In [ ]:
# CHẠY CELL NÀY TRƯỚC; SAU ĐÓ RESTART SESSION MỘT LẦN
# Cập nhật thư viện và thay trực tiếp lời gọi datetime.utcnow() lỗi thời
import sys, subprocess, pathlib, importlib.util

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'jupyter_client', 'ipykernel', 'python-calamine', 'mlxtend'], check=True)

spec = importlib.util.find_spec('jupyter_client')
if spec is None or not spec.submodule_search_locations:
    raise RuntimeError('Không tìm thấy jupyter_client sau khi cài đặt')
session_file = pathlib.Path(next(iter(spec.submodule_search_locations))) / 'session.py'
source = session_file.read_text(encoding='utf-8')
old = 'datetime.utcnow().replace(tzinfo=utc)'
new = 'datetime.now(__import__("datetime").timezone.utc)'
if old in source:
    try:
        session_file.write_text(source.replace(old, new), encoding='utf-8')
    except PermissionError as exc:
        raise RuntimeError('Không có quyền sửa thư viện trong runtime; cần dùng môi trường khác') from exc
    print('Đã thay datetime.utcnow() thành datetime.now(timezone.utc) trong:', session_file)
elif new in source or 'datetime.now(timezone.utc)' in source:
    print('Thư viện đã dùng datetime.now(timezone.utc), không cần sửa.')
else:
    print('Không tìm thấy mẫu utcnow cũ; thư viện có thể đã thay đổi. Kiểm tra:', session_file)
print('QUAN TRỌNG: Runtime > Restart session, sau đó chạy từ CELL 2 trở đi.')
print('Nếu cảnh báo do máy chủ Colab bên ngoài runtime phát ra, notebook không sửa được tiến trình đó.')


In [ ]:
import io
import time
import requests
import pandas as pd
import inspect
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

STUDENT_INFO = "Thông tin sinh viên: Lê Nguyễn Anh Dự - 6451071011"
sid = "12q4inbGbTS1UKs9sHA1X87vSRV8feEHV"
url = f"https://docs.google.com/spreadsheets/d/{sid}/export?format=xlsx"

print(f"Đang tải Online Retail | {STUDENT_INFO}", flush=True)
t0 = time.perf_counter()
response = requests.get(url, timeout=(15, 120))
response.raise_for_status()
if not response.content.startswith(b"PK"):
    raise ValueError("Nội dung tải về không phải XLSX. Hãy kiểm tra quyền Google Sheets.")
print(f"Tải xong {len(response.content) / (1024**2):.2f} MB trong {time.perf_counter()-t0:.1f} giây", flush=True)

print("Đang đọc dữ liệu Excel bằng calamine...", flush=True)
t0 = time.perf_counter()
df = pd.read_excel(io.BytesIO(response.content), engine="calamine")
print(f"Đã đọc {len(df)} dòng, {df.shape[1]} cột trong {time.perf_counter()-t0:.1f} giây | {STUDENT_INFO}", flush=True)
display(df.head())


## Bước 1. Làm sạch và loại hóa đơn tín dụng


In [ ]:
df["Description"]=df["Description"].astype("string").str.strip().str.replace(r"\s+"," ",regex=True)
df["InvoiceNo"]=df["InvoiceNo"].astype("string").str.strip()
df=df.dropna(subset=["InvoiceNo","Description"]).copy()
df=df[df["InvoiceNo"].ne("") & df["Description"].ne("")].copy()
df["InvoiceNo"]=df["InvoiceNo"].astype(str)
credit=df["InvoiceNo"].str.contains("C",case=False,na=False)
n_credit=df.loc[credit,"InvoiceNo"].nunique()
df=df.loc[~credit].copy()
print(f"Hóa đơn tín dụng đã loại: {n_credit}; số dòng còn lại: {len(df)} | {STUDENT_INFO}")


## Bước 2. Lọc Germany, gom nhóm và one-hot encoding


In [ ]:
MIN_SUP=0.05; MIN_CONF=0.50
de=df.loc[df["Country"].eq("Germany")]
grouped=de.groupby(["InvoiceNo","Description"],as_index=False)["Quantity"].sum()
positive=grouped[grouped["Quantity"]>0]
transactions=positive.groupby("InvoiceNo")["Description"].agg(lambda x:sorted(set(x))).tolist()
te=TransactionEncoder()
basket=pd.DataFrame(te.fit(transactions).transform(transactions),columns=te.columns_,dtype=bool).drop(columns=["POSTAGE"],errors="ignore")
print(f"Germany: {len(de)} dòng; {len(transactions)} hóa đơn; {basket.shape[1]} mặt hàng | {STUDENT_INFO}")
print(f"Mẫu ma trận giao dịch | {STUDENT_INFO}"); display(basket.iloc[:5,:12])


## Bước 3. Apriori và luật kết hợp


In [ ]:
frequent_a=apriori(basket,min_support=MIN_SUP,use_colnames=True).sort_values("support",ascending=False,kind="stable").reset_index(drop=True)
def rules_from(itemsets,n,confidence):
    if itemsets.empty or not itemsets["itemsets"].map(len).ge(2).any():
        return pd.DataFrame(columns=["antecedents","consequents","support","confidence","lift"])
    kw={"metric":"confidence","min_threshold":confidence}
    if "num_itemsets" in inspect.signature(association_rules).parameters: kw["num_itemsets"]=n
    return association_rules(itemsets,**kw)
rules_a=rules_from(frequent_a,len(basket),MIN_CONF)
if not rules_a.empty: rules_a=rules_a.sort_values(["confidence","support"],ascending=False).reset_index(drop=True)
print(f"Tập phổ biến Apriori, min_sup=5% ({len(frequent_a)}) | {STUDENT_INFO}"); display(frequent_a)
print(f"Luật Apriori, min_conf=50% ({len(rules_a)}) | {STUDENT_INFO}"); display(rules_a)


## Bước 4. FP-Growth và so sánh


In [ ]:
frequent_f=fpgrowth(basket,min_support=MIN_SUP,use_colnames=True).sort_values("support",ascending=False,kind="stable").reset_index(drop=True)
rules_f=rules_from(frequent_f,len(basket),MIN_CONF)
if not rules_f.empty: rules_f=rules_f.sort_values(["confidence","support"],ascending=False).reset_index(drop=True)
sa=set(frequent_a.itemsets.map(frozenset)); sf=set(frequent_f.itemsets.map(frozenset))
ra=set(zip(rules_a.antecedents.map(frozenset),rules_a.consequents.map(frozenset))) if not rules_a.empty else set()
rf=set(zip(rules_f.antecedents.map(frozenset),rules_f.consequents.map(frozenset))) if not rules_f.empty else set()
comparison=pd.DataFrame({"Chỉ số":["Tập Apriori","Tập FP-Growth","Tập trùng nhau","Luật Apriori","Luật FP-Growth","Luật trùng nhau"],"Số lượng":[len(sa),len(sf),len(sa&sf),len(ra),len(rf),len(ra&rf)]})
print(f"Tập phổ biến FP-Growth ({len(frequent_f)}) | {STUDENT_INFO}"); display(frequent_f)
print(f"Luật FP-Growth ({len(rules_f)}) | {STUDENT_INFO}"); display(rules_f)
print(f"So sánh thuật toán tại Germany | {STUDENT_INFO}"); display(comparison)
print(f"Với cùng dữ liệu/ngưỡng, hai thuật toán cho cùng tập phổ biến: {sa==sf} | {STUDENT_INFO}")
